<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# When not to build a model

**[When not to build a model](https://learning.nextia-ai.com/courses/forecasting/m05/when-not-to-build-a-model/)** · Forecasting and Time Series: Predict What Comes Next · Module 5, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read the backtest table by cost and by fit time as well as by error, check product by product whether a complex model wins, and check every number in a forecast note that an LLM drafted.

| | |
|---|---|
| **Time** | About 30 minutes, of which about 2 minutes of waiting for the backtests |
| **Needs** | An internet connection for the setup cells. No account and no key. No model calls: the notes were recorded once. |
| **You should know** | The backtest table of this module, from [Gradient boosting with LightGBM](https://learning.nextia-ai.com/courses/forecasting/m05/gradient-boosting-with-lightgbm/) and [Deep and foundation models](https://learning.nextia-ai.com/courses/forecasting/m05/deep-and-foundation-models/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m05/when-not-to-build-a-model/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M05-L04-when-not-to-build-a-model/when-not-to-build-a-model-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, lightgbm 4.7.0, scikit-learn 1.9.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 mlforecast==1.1.0 lightgbm==4.7.0 scikit-learn==1.9.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)
import mlforecast; print("mlforecast", mlforecast.__version__)
import lightgbm; print("lightgbm", lightgbm.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The table, sorted three ways

The next cell holds the course's backtest table for the orders, as measured in the earlier lessons (50 products, 13 cut-offs, 14 days). It prints the five best rows by WAPE, by cost and by fit time.

> **Predict.** Is the method with the lowest WAPE also the one with the lowest cost? Then run the cell.

In [ ]:
table = pd.DataFrame([
    # method, MASE, WAPE %, bias %, fit seconds (13 cut-offs), cost $ per order
    ("moving average (28 days)", 0.875, 43.5, -0.1, 0.11, 19444),
    ("seasonal naive", 1.034, 51.3, -0.5, 0.05, 16739),
    ("naive", 1.065, 53.2, 7.4, 0.02, 28261),
    ("drift", 1.075, 53.8, 7.8, 0.04, 28570),
    ("ETS", 0.832, 41.3, 0.1, 46.04, 16578),
    ("AutoARIMA", 0.834, 41.4, 1.4, 995.77, 16257),
    ("Theta", 0.837, 41.5, 0.1, 42.17, 16812),
    ("linear regression, global", 0.866, 42.6, -0.1, 0.33, 17264),
    ("linear regression, per product", 0.835, 41.7, -1.4, 13.19, 15137),
    ("LightGBM, recursive", 0.816, 40.7, 0.7, 16.42, 14749),
    ("LightGBM, direct", 0.813, 40.7, -0.6, 156.19, 13711),
    ("LightGBM, lags >= 14", 0.828, 40.9, -0.7, 11.55, 14629),
    ("Chronos-Bolt", 0.784, 38.7, -5.6, 1.82, 14750),
], columns=["method", "MASE", "WAPE %", "bias %", "fit s", "cost $"])
for column in ["WAPE %", "cost $", "fit s"]:
    print(f"best by {column}:", ", ".join(f"{m} ({v:,})" for m, v in table.sort_values(column).head(5)[["method", column]].values))
cheapest = table["cost $"].min()
table["extra cost $"] = table["cost $"] - cheapest
print(table.sort_values("cost $")[["method", "WAPE %", "fit s", "cost $", "extra cost $"]].to_string(index=False))

> **Check.** You should see Chronos-Bolt first by WAPE (38.7), LightGBM, direct first by cost (13,711) and the naive forecast first by fit time. In the full table, the seasonal naive forecast costs 3,028 dollars more per order than the cheapest method.

The ranking changes with the question. Kenji pays for the order, so the cost column decides for him. But the fit time, the people who must understand the method, and the work to run it count too.

## 2. Product by product

The table adds up 50 products. A method that wins in total can lose on many products. The next cell runs ETS and LightGBM (recursive) at the 13 cut-offs, then computes the WAPE of each product for each method.

> **Predict.** On how many of the 50 products does ETS beat LightGBM? Then run the cell (about a minute).

In [ ]:
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

cal = pd.read_csv("data/calendar.csv", parse_dates=["date"])
cal["black_friday_week"] = 0
for day in cal.loc[cal["holiday"] == "Black Friday", "date"]:
    cal.loc[(cal["date"] >= day) & (cal["date"] <= day + pd.Timedelta(days=3)), "black_friday_week"] = 1
cal["christmas_run_up"] = ((cal["date"].dt.month == 12) & (cal["date"].dt.day <= 23)).astype(int)
EXOG = ["price", "on_promotion", "is_holiday", "black_friday_week", "christmas_run_up"]
sales = load_sales(BACKTEST_PRODUCTS).merge(
    cal[["date", "is_holiday", "black_friday_week", "christmas_run_up"]].rename(columns={"date": "ds"}), on="ds", how="left")
LGB = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=20, subsample=0.8,
           subsample_freq=1, colsample_bytree=0.8, random_state=35, n_jobs=2, verbose=-1)

def lightgbm(train, h, future):
    f = MLForecast(models={"lgbm": lgb.LGBMRegressor(**LGB)}, freq="D", lags=[1, 7, 14, 28],
                   lag_transforms={1: [RollingMean(7), RollingMean(28)], 7: [RollingMean(4)]},
                   date_features=["dayofweek", "month", "dayofyear"], num_threads=1)
    f.fit(train[["unique_id", "ds", "y"] + EXOG], static_features=[])
    p = f.predict(h, X_df=future[["unique_id", "ds"] + EXOG])
    return p.rename(columns={"lgbm": "yhat"}).assign(yhat=lambda d: d["yhat"].clip(lower=0))

def ets(train, h, future):
    out = StatsForecast(models=[AutoETS(season_length=7)], freq="D").forecast(df=train[["unique_id", "ds", "y"]], h=h)
    return out.rename(columns={"AutoETS": "yhat"}).assign(yhat=lambda d: d["yhat"].clip(lower=0))

forecasts = {"seasonal naive": bt.backtest(sales, bt.seasonal_naive(7), h=14, origins=ORIGINS),
             "ETS": bt.backtest(sales, ets, h=14, origins=ORIGINS),
             "LightGBM": bt.backtest(sales, lightgbm, h=14, origins=ORIGINS)}

def wape_per_product(fc):
    e = fc.assign(miss=(fc["y"] - fc["yhat"]).abs()).groupby("unique_id")[["miss", "y"]].sum()
    return 100 * e["miss"] / e["y"]

per_product = pd.DataFrame({name: wape_per_product(fc) for name, fc in forecasts.items()}).round(1)
print("ETS beats LightGBM on", int((per_product["ETS"] < per_product["LightGBM"]).sum()), "of", len(per_product), "products")
print(per_product.loc[["P013", "P025", "P075", "P077"]])

> **Check.** You should see `ETS beats LightGBM on 21 of 50 products`, and four rows: on the parasol P025, LightGBM has a WAPE of 49.3% against 65.1% for ETS.

LightGBM wins in total, but ETS is better on 21 products. A team could keep ETS for those products, or keep one method for all and accept the loss: a second method is a second thing to run and watch.

## 3. Your turn: the simplest method

> **Your turn.** Count the products where the **seasonal naive** forecast has a lower WAPE than LightGBM in `per_product`. Put the count in `naive_wins` as a Python integer. Then run the check cell.

In [ ]:
naive_wins = ...  # the number of products where the seasonal naive WAPE is lower than LightGBM's

In [ ]:
expect('naive_wins', naive_wins, 3)

The forecast "the same weekday last week" beats a 400-tree model on 3 of 50 products. On the others, LightGBM's lead is what you pay its cost for.

## 4. Check the numbers in a forecast note

The forecasting job of Module 8 writes a short note for the operations manager every week. A small hosted LLM (Azure `chat-small`) drafted 10 notes from the numbers of 10 weekly plans. The course recorded them once, so this notebook makes no model calls. The next cell downloads the recordings and checks each note: every number in the text must be one of the numbers that the LLM received.

> **Predict.** The LLM was told to use only the numbers it got. Will all 10 notes pass? Then run the cell.

In [ ]:
import re

fetch("notes_recorded.json", LABS + "forecast-job/end-of-m08/planner/notes_recorded.json",
      Path("notes_recorded.json"), "44843cdb31426e92fa6b2a8788da9039bcd77e53189870bae2b3164dd6a2e2c5")
notes = json.loads(Path("notes_recorded.json").read_text())["notes"]

NUMBER = re.compile(r"(?<![\w.])[-+]?\d[\d,]*(?:\.\d+)?%?")   # 12,839 or -23% or 0.5

def value(text):
    return float(text.rstrip("%").replace(",", "").lstrip("+"))

def check(note, facts):
    """The numbers in the note that are not in the facts (within 1%). 1, 2, 7 and 14 are always allowed."""
    allowed = {value(m) for m in NUMBER.findall(json.dumps(facts))} | {1.0, 2.0, 7.0, 14.0}
    return [m for m in NUMBER.findall(note) if not any(abs(value(m) - a) <= 0.01 * max(abs(a), 1) for a in allowed)]

for n in notes:
    bad = check(n["note"], n["facts_sent"])
    print(n["week"], "passes" if not bad else f"NOT SUPPORTED: {bad}")

> **Check.** You should see 9 notes that pass, and `2025-03-17 NOT SUPPORTED: ['23%', '56%']`.

The next cell prints the sentence and the alarm it came from.

In [ ]:
n = next(x for x in notes if x["week"] == "2025-03-17")
print("ALARM:", n["facts_sent"]["alarms"][0][:115])
print("NOTE: ", next(s for s in n["note"].split(". ") if "23%" in s))

> **Check.** The alarm says `(-23% and -56%`; the note says `by 23% and 56%`.

The note dropped the minus signs. Here the words "too low" still carry the meaning, but the check cannot know that, and in another sentence a dropped sign turns a shortfall into a surplus. The fix is simple: copy the plan's numbers as they are, or flag the note for a person.

The check has a limit too. Print the first sentence of the note of 2025-03-10: it passes, because 12,839 is in the plan. Read what the number counts.

In [ ]:
print(notes[0]["note"].split(". ")[0])
print("in the plan:", {k: v for k, v in notes[0]["facts_sent"].items() if k.startswith("order")})

> **Check.** You should see `… to handle 12,839 orders …` and `order_units: 12839`.

12,839 is the number of **units** to order, not customer orders. A check of numbers does not check meaning. A person still reads the note before it goes out. And the LLM never computes the forecast: the numbers come from the backtested job.

## Next

You read the table for cost, speed and trust, and checked an LLM's note against the plan. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m05/when-not-to-build-a-model/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Hierarchies that must add up](https://learning.nextia-ai.com/courses/forecasting/m06/hierarchies-that-must-add-up/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m05/when-not-to-build-a-model/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.